# Phase 1 — Explore the SECOM data

**Goal:** understand the semiconductor data before modeling.

Dataset → Load → Explore → Clean → Visualize

Each row is one wafer: a pass/fail label (`class`), a `timestamp`, and 590 anonymous sensor readings (`Attribute 1` … `Attribute 590`).

## 1. Load

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("../data")
RAW_PATH = DATA_DIR / "secom_raw.csv"

# Download the SECOM data from UCI once, then reuse the local copy
data_url = "https://archive.ics.uci.edu/static/public/179/data.csv"

if RAW_PATH.exists():
    df = pd.read_csv(RAW_PATH)
else:
    df = pd.read_csv(data_url)
    df.to_csv(RAW_PATH, index=False)

df = df.copy()  # consolidate memory so adding columns later doesn't trigger warnings

print(df.shape, "(Rows/ Manufactured wafers, Columns/ label + timestamp + sensor measurements)")

In [ ]:
df.head()

In [ ]:
df.info()

## 2. Explore

### Pass / fail balance
In SECOM, `-1` means the wafer **passed** and `1` means it **failed**. We add a clearer 0/1 `fail` column.

In [ ]:
df["fail"] = (df["class"] == 1).astype(int)

counts = df["fail"].value_counts().rename({0: "pass", 1: "fail"})
print(counts)
print(f"\nFail rate: {df['fail'].mean():.1%}")

Only about 1 in 15 wafers fails, so the classes are **heavily imbalanced**. A model that always predicts "pass" would be ~93% accurate and completely useless, so in Phase 2 we should judge models on fail recall, precision, F1, or ROC-AUC rather than plain accuracy.

### Timestamps

In [ ]:
# Dates are day-first (DD/MM/YYYY), and some rows omit seconds or leading zeros
df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed", dayfirst=True)

print("First wafer:", df["timestamp"].min())
print("Last wafer: ", df["timestamp"].max())
print("Span:       ", df["timestamp"].max() - df["timestamp"].min())

### Sensor summary statistics

In [ ]:
sensor_cols = [c for c in df.columns if c.startswith("Attribute")]
print(len(sensor_cols), "sensor columns")

df[sensor_cols].describe().T.head(15)

The sensors are on very different scales (some in the thousands, some below 1), so they will need scaling before models like logistic regression.

### Missing values

In [ ]:
missing_frac = df[sensor_cols].isna().mean().sort_values(ascending=False)

print("Total missing cells:     ", df[sensor_cols].isna().sum().sum())
print("Sensors with any missing:", (missing_frac > 0).sum())
print("Sensors >50% missing:    ", (missing_frac > 0.5).sum())

missing_frac.head(10)

### Constant sensors
A sensor that reports the same value for every wafer carries no information.

In [ ]:
n_unique = df[sensor_cols].nunique()
constant_cols = n_unique[n_unique <= 1].index.tolist()
print(len(constant_cols), "constant sensors")

## 3. Visualize

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class balance
counts.plot.bar(ax=axes[0], color=["tab:blue", "tab:red"], rot=0)
axes[0].set_title("Wafers by outcome")
axes[0].set_ylabel("Number of wafers")

# How much of each sensor is missing
axes[1].hist(missing_frac, bins=50, color="tab:gray")
axes[1].axvline(0.5, color="tab:red", linestyle="--", label="50% cutoff")
axes[1].set_title("Missing values per sensor")
axes[1].set_xlabel("Fraction of wafers missing")
axes[1].set_ylabel("Number of sensors")
axes[1].legend()

plt.tight_layout()
plt.show()

### Failures over time
If failures cluster in certain weeks, the process may have drifted (a useful hint for Phase 4, anomaly detection).

In [ ]:
weekly = df.set_index("timestamp").resample("W")["fail"].agg(["count", "mean"])

fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.bar(weekly.index, weekly["count"], width=5, color="tab:gray", alpha=0.5, label="wafers produced")
ax1.set_ylabel("Wafers per week")

ax2 = ax1.twinx()
ax2.plot(weekly.index, weekly["mean"], color="tab:red", marker="o", label="fail rate")
ax2.set_ylabel("Fail rate")

ax1.set_title("Weekly production volume and fail rate")
fig.legend(loc="upper center", bbox_to_anchor=(0.5, 0.88))
plt.tight_layout()
plt.show()

## 4. Clean

Cleaning steps:
1. Drop sensors that are **more than 50% missing**. Too little data to trust.
2. Drop **constant** sensors. No information.
3. Drop the original `class` column (replaced by `fail`).

We do **not** fill in (impute) the remaining missing values here. Imputation and scaling should be learned from the training data only, inside the Phase 2 model pipeline, otherwise information from the test set leaks into training.

In [ ]:
high_missing_cols = missing_frac[missing_frac > 0.5].index.tolist()
drop_cols = sorted(set(high_missing_cols) | set(constant_cols), key=sensor_cols.index)

clean = df.drop(columns=drop_cols + ["class"])
kept_sensors = [c for c in clean.columns if c.startswith("Attribute")]

print(f"Dropped {len(high_missing_cols)} high-missing + {len(constant_cols)} constant "
      f"= {len(drop_cols)} sensors")
print(f"Kept {len(kept_sensors)} of {len(sensor_cols)} sensors")
print(f"Remaining missing cells: {clean[kept_sensors].isna().sum().sum()} "
      f"({clean[kept_sensors].isna().mean().mean():.1%} of sensor values)")

## 5. Which sensors differ between pass and fail?

A first look at which sensors might matter. For each sensor we compare the pass and fail groups using the difference in medians, divided by the sensor's overall spread (so sensors on different scales are comparable). Medians are used because many sensors have extreme outliers.

In [ ]:
pass_med = clean.loc[clean["fail"] == 0, kept_sensors].median()
fail_med = clean.loc[clean["fail"] == 1, kept_sensors].median()
spread = clean[kept_sensors].std()

effect = ((fail_med - pass_med) / spread).dropna()
top_sensors = effect.abs().sort_values(ascending=False).head(10).index

effect[top_sensors].round(3)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

for ax, col in zip(axes.flat, top_sensors[:6]):
    data = [clean.loc[clean["fail"] == 0, col].dropna(),
            clean.loc[clean["fail"] == 1, col].dropna()]
    ax.boxplot(data, tick_labels=["pass", "fail"], showfliers=False)
    ax.set_title(col)

fig.suptitle("Sensors with the biggest pass/fail difference (outliers hidden)")
plt.tight_layout()
plt.show()

### How correlated are the sensors?
Many sensors likely measure related things. Strong correlation means there is redundant information we can trim in Phase 3.

In [ ]:
corr = clean[kept_sensors].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))

print("Sensor pairs with |correlation| > 0.95:", int((upper > 0.95).sum().sum()))

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="viridis", vmin=0, vmax=1)
ax.set_title("Absolute correlation between kept sensors")
ax.set_xticks([])
ax.set_yticks([])
fig.colorbar(im, ax=ax)
plt.show()

## 6. Save the cleaned data for Phase 2

In [ ]:
CLEAN_PATH = DATA_DIR / "secom_clean.csv"
clean.to_csv(CLEAN_PATH, index=False)
print("Saved", clean.shape, "to", CLEAN_PATH)

## Phase 1 takeaways

- **1,567 wafers, 590 sensors**, collected July–October 2008.
- **Heavily imbalanced:** only ~6.6% of wafers fail. Use recall, F1, or ROC-AUC, not accuracy, and consider class weights.
- **Messy sensors:** most sensors have some missing values, and a few are mostly empty. Many are constant. After cleaning, we keep the remaining sensors in `data/secom_clean.csv`.
- **Scales vary wildly**, so standardize features for linear models.
- **Failures were not steady over time:** the weekly fail rate was much higher in July–August than in September–October. That could be process drift or early ramp-up, and it matters for Phase 4 (anomaly detection). It also means a time-based train/test split is a tougher, more realistic test than a random one.
- **Lots of redundancy** between sensors, which is a good target for feature selection in Phase 3.
- **Next (Phase 2):** split train/test (stratified by `fail`), then a pipeline of median imputation → scaling → simple classifier (e.g. logistic regression with `class_weight="balanced"`).